# Week 5: Nonlinear Stage-2 ML Modeling Benchmark

**Project:** Query-Conditioned Context Allocator (QCCA) for Efficient Retrieval-Augmented Generation  
**Parent Architecture:** SARA — Selective and Adaptive Retrieval-augmented Generation with Context Compression  
**Experiment:** Week 5 Confirmatory Benchmark of Nonlinear Stage-2 Classifiers  
**Evaluation Population:** QASPER Development Split (86 papers, 231 questions)  
**Governance:** Strict Paper-Disjoint GroupKFold. Held-out test set remains 100% frozen & untouched.  

---

### Research Question:
> **"Can a small nonlinear model (Decision Tree, Random Forest, XGBoost, LightGBM, HistGradientBoosting, or a tiny MLP) learn the validated $4 \rightarrow 6$ expansion decision better than Balanced Logistic Regression, particularly by curbing false expansions caused by high keyword density + redundant evidence, while preserving recall on high-gain expansions?"**

## 1. Environment Verification & Library Imports

In [1]:
import os
import sys
from pathlib import Path

# Robust project root discovery
_curr = Path.cwd().resolve()
candidates = [_curr, _curr.parent, _curr.parent.parent]
PROJECT_ROOT = next((p for p in candidates if (p / "src").is_dir() and (p / "results").is_dir()), _curr)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import sklearn
import torch
import xgboost as xgb
import lightgbm as lgb

from src.week5.model_benchmark import (
    CONTEXT_TOKEN_COSTS,
    DEFAULT_THRESHOLDS,
    STAGE_2_CORE_FEATURES,
    STAGE_2_SUBSET_FEATURES,
    load_week5_dev_data,
    simulate_policy_c1,
    train_eval_model_oof,
    evaluate_model_threshold_grid,
    run_paper_clustered_paired_bootstrap,
    analyze_feature_interactions,
)
from src.week4.qcca_v2 import compute_pareto_frontier

print(f"Project root: {PROJECT_ROOT}")
print(f"Scikit-learn version: {sklearn.__version__}")
print(f"PyTorch version: {torch.__version__}")
print(f"XGBoost version: {xgb.__version__}")
print(f"LightGBM version: {lgb.__version__}")
print("Libraries imported successfully.")

Project root: /home/gunavenkat/Downloads/CS787-RAG-Project
Scikit-learn version: 1.9.1
PyTorch version: 2.14.0+cu130
XGBoost version: 3.0.5
LightGBM version: 4.7.0
Libraries imported successfully.


/home/gunavenkat/Downloads/CS787-RAG-Project/sara_env/lib/python3.11/site-packages/xgboost/core.py:377: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc >= 2.28) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(


## 2. Load Development Split & Stage-2 Transition Targets
Loads development features, per-query response surfaces, and constructs the marginal gain $G_{4 \rightarrow 6} = F_1(k=6) - F_1(k=4)$ and target $Y_{4 \rightarrow 6} = \mathbb{I}[G_{4 \rightarrow 6} > 0.01]$.

In [2]:
df_f, df_t, df_merged = load_week5_dev_data(
    feature_matrix_path=str(PROJECT_ROOT / "results/week4/feature_discovery/feature_matrix.csv"),
    dev_matrix_path=str(PROJECT_ROOT / "results/week2/processed/dev_per_query_k_matrix.csv"),
    oracle_path=str(PROJECT_ROOT / "results/week3/processed/epsilon_oracle_all.csv"),
)

n_queries = len(df_merged)
n_papers = df_merged["paper_id"].nunique()
n_pos = int(df_merged["Y_4_6"].sum())

print(f"Loaded {n_queries} queries across {n_papers} papers.")
print(f"Stage 2 positive transitions: {n_pos} ({n_pos / n_queries * 100:.2f}% base rate)")
print(f"Validated 5 Stage-2 core features: {STAGE_2_CORE_FEATURES}")
df_merged[["question_id", "paper_id", "F1_k4", "F1_k6", "G_4_6", "Y_4_6"] + STAGE_2_CORE_FEATURES].head()

Loaded 231 queries across 86 papers.
Stage 2 positive transitions: 45 (19.48% base rate)
Validated 5 Stage-2 core features: ['bm25_mean_score', 'lex_coverage_top2', 'sem_sim_mean_top6', 'evidence_query_cluster_span', 'evidence_lexical_overlap_mean']


,question_id,paper_id,F1_k4,F1_k6,G_4_6,Y_4_6,bm25_mean_score,lex_coverage_top2,sem_sim_mean_top6,evidence_query_cluster_span,evidence_lexical_overlap_mean
0,1912,638,0.0000,0.0000,0.0000,0,0.054874,0.250000,0.584200,1,0.001127
1,2201,798,0.6316,0.0000,-0.6316,0,2.393718,0.571429,0.721364,4,0.134612
2,883,254,0.6667,1.0000,0.3333,1,0.404775,1.000000,0.617398,3,0.136589
3,884,254,1.0000,0.8889,-0.1111,0,0.153224,0.500000,0.536448,1,0.139618
4,104,28,0.0000,0.0000,0.0000,0,0.594406,0.500000,0.556276,4,0.141486


## 3. Phase 1: Bit-for-Bit Reproduction of Balanced Logistic Regression Baseline
Must reproduce:
- ROC-AUC ~0.6503
- PR-AUC ~0.3199
- Balanced Accuracy ~0.6448
- Policy F1 ~0.3842
- Mean k ~4.82
- Context Token Reduction vs k=8 ~36.54%

In [3]:
res_base_lr = train_eval_model_oof(
    df_merged=df_merged,
    model_family="logistic_regression",
    feature_cols=STAGE_2_CORE_FEATURES,
    n_outer_splits=5,
    n_inner_splits=3,
    calibrate=False,
    seed=42,
)

print("=" * 60)
print("PHASE 1 BASELINE REPRODUCTION RESULTS")
print("=" * 60)
print(f"ROC-AUC:           {res_base_lr['roc_auc']:.4f} (Expected: ~0.6503)")
print(f"PR-AUC:            {res_base_lr['pr_auc']:.4f} (Expected: ~0.3199)")
print(f"Balanced Accuracy: {res_base_lr['balanced_acc']:.4f} (Expected: ~0.6448)")
print(f"Precision:         {res_base_lr['precision']:.4f}")
print(f"Recall:            {res_base_lr['recall']:.4f}")
print(f"Brier Score:       {res_base_lr['brier_score']:.4f}")
print("-" * 60)
sm = res_base_lr['policy_summary']
print(f"Policy Mean F1:    {sm['mean_f1']:.4f} (Expected: ~0.3842)")
print(f"Mean Budget k:     {sm['mean_k']:.2f} (Expected: ~4.82)")
print(f"Context Tokens:    {sm['mean_context_tokens']:.1f} (Expected: ~1115.9)")
print(f"Token Reduction:   {sm['context_reduction_pct_vs_k8']:.2f}% (Expected: ~36.54%)")
print(f"Mean Regret:       {sm['mean_regret']:.4f}")
print(f"Zero-Regret %:     {sm['fraction_zero_regret']:.1f}%")
print(f"False Expansion %: {sm['false_expansion_pct']:.2f}% (66 queries)")
print(f"False Stop %:      {sm['false_stop_pct']:.2f}% (16 queries)")
print("=" * 60)

PHASE 1 BASELINE REPRODUCTION RESULTS
ROC-AUC:           0.6503 (Expected: ~0.6503)
PR-AUC:            0.3199 (Expected: ~0.3199)
Balanced Accuracy: 0.6448 (Expected: ~0.6448)
Precision:         0.3053
Recall:            0.6444
Brier Score:       0.2307
------------------------------------------------------------
Policy Mean F1:    0.3842 (Expected: ~0.3842)
Mean Budget k:     4.82 (Expected: ~4.82)
Context Tokens:    1115.9 (Expected: ~1115.9)
Token Reduction:   36.54% (Expected: ~36.54%)
Mean Regret:       0.1053
Zero-Regret %:     69.7%
False Expansion %: 28.57% (66 queries)
False Stop %:      6.93% (16 queries)


## 4. Phase 2: Controlled Nonlinear Model Benchmark
Evaluates candidate models across identical paper-disjoint GroupKFold splits:
1. Balanced Logistic Regression (5 feats, Baseline)
2. Balanced Logistic Regression (2 feats: BM25 + Lexical Coverage)
3. Shallow Decision Tree (depths 2-5, balanced)
4. Random Forest (shallow, balanced)
5. XGBoost (small-data regularized, 5 feats)
6. XGBoost (2 feats: BM25 + Lexical Coverage)
7. LightGBM (small-data regularized)
8. HistGradientBoosting (balanced)
9. Small MLP (PyTorch, class-weighted BCE, early stopping)
10. Calibrated variants (Platt/sigmoid)

In [4]:
from src.week5.run_benchmark import run_full_week5_benchmark

# Execute the full leak-free benchmark suite
df_model_comp, df_boot, df_policy_comp = run_full_week5_benchmark()
display(df_model_comp[["model_name", "roc_auc", "pr_auc", "balanced_acc", "policy_f1_t050", "mean_k_t050", "token_reduction_t050", "false_expansion_pct_t050", "false_stop_pct_t050"]])

2026-09-30 22:41:39,934 [INFO] ================================================================================
2026-09-30 22:41:39,935 [INFO] STARTING WEEK 5: NONLINEAR STAGE-2 ML MODELING BENCHMARK
2026-09-30 22:41:39,936 [INFO] ================================================================================
2026-09-30 22:41:39,953 [INFO] Loaded 231 queries across 86 papers. Positive transitions: 45 (19.5%)
2026-09-30 22:41:39,954 [INFO] Evaluating: Balanced Logistic Regression (5 feats, Baseline) ...
2026-09-30 22:41:40,168 [INFO] Evaluating: Balanced Logistic Regression (2 feats: BM25+Lex) ...
2026-09-30 22:41:40,361 [INFO] Evaluating: Shallow Decision Tree (5 feats) ...
2026-09-30 22:41:41,184 [INFO] Evaluating: Random Forest (5 feats) ...
2026-09-30 22:41:48,090 [INFO] Evaluating: XGBoost (5 feats) ...
2026-09-30 22:41:49,124 [INFO] Evaluating: XGBoost (2 feats: BM25+Lex) ...
2026-09-30 22:41:50,114 [INFO] Evaluating: LightGBM (5 feats) ...
2026-09-30 22:41:51,054 [INFO] Evaluati

,model_name,roc_auc,pr_auc,balanced_acc,policy_f1_t050,mean_k_t050,token_reduction_t050,false_expansion_pct_t050,false_stop_pct_t050
0,"Balanced Logistic Regression (5 feats, Baseline)",0.6503,0.3199,0.6448,0.3842,4.82,36.54,28.57,6.93
1,Balanced Logistic Regression (2 feats: BM25+Lex),0.6639,0.3664,0.6145,0.3800,4.83,36.44,29.87,7.79
2,Shallow Decision Tree (5 feats),0.5197,0.2093,0.5219,0.3671,4.64,38.67,25.11,12.55
3,Random Forest (5 feats),0.6363,0.2962,0.6192,0.3717,4.73,37.66,25.54,8.66
4,XGBoost (5 feats),0.6265,0.2870,0.6151,0.3725,4.56,39.59,19.05,10.39
5,XGBoost (2 feats: BM25+Lex),0.6335,0.3293,0.6023,0.3780,4.69,38.06,24.68,9.52
6,LightGBM (5 feats),0.5803,0.2825,0.5616,0.3632,4.78,37.05,29.44,9.96
7,HistGradientBoosting (5 feats),0.6130,0.2980,0.5970,0.3716,4.71,37.86,25.54,9.52
8,"Small MLP (5->8->1, 5 feats)",0.5755,0.2616,0.5606,0.3638,4.65,38.57,24.24,11.26
9,"Calibrated Logistic Regression (Platt, 5 feats)",0.5838,0.2510,0.5000,0.3596,4.00,46.20,0.00,19.48


## 5. Phase 3: Paper-Clustered Paired Bootstrap ($B=2,000$)
Rigorous statistical comparison against Balanced Logistic Regression and Static $k=8$.

In [5]:
df_boot_lr = df_boot[df_boot["baseline"] == "Balanced Logistic Regression (5 feats, Baseline)"].copy()
print("Paired Clustered Bootstrap Comparisons vs Balanced Logistic Regression (B=2,000):")
display(df_boot_lr[["candidate", "mean_delta_f1", "ci_95_f1_low", "ci_95_f1_high", "statistically_detectable_f1", "mean_delta_tokens", "ci_95_tokens_low", "ci_95_tokens_high"]])

Paired Clustered Bootstrap Comparisons vs Balanced Logistic Regression (B=2,000):


,candidate,mean_delta_f1,ci_95_f1_low,ci_95_f1_high,statistically_detectable_f1,mean_delta_tokens,ci_95_tokens_low,ci_95_tokens_high
2,Balanced Logistic Regression (2 feats: BM25+Lex),-0.0042,-0.0177,0.0091,False,1.9,-16.3,20.8
5,Shallow Decision Tree (5 feats),-0.0168,-0.0348,0.0001,False,-37.6,-66.5,-9.3
8,Random Forest (5 feats),-0.0124,-0.0300,0.0038,False,-19.3,-42.6,5.6
11,XGBoost (5 feats),-0.0115,-0.0313,0.0077,False,-53.4,-77.8,-29.8
14,XGBoost (2 feats: BM25+Lex),-0.0057,-0.0277,0.0167,False,-26.5,-58.8,4.4
17,LightGBM (5 feats),-0.0210,-0.0381,-0.0074,True,-8.5,-34.4,20.0
20,HistGradientBoosting (5 feats),-0.0124,-0.0323,0.0071,False,-23.1,-49.5,5.3
23,"Small MLP (5->8->1, 5 feats)",-0.0203,-0.0391,-0.0042,True,-35.5,-61.5,-10.6
26,"Calibrated Logistic Regression (Platt, 5 feats)",-0.0242,-0.0516,0.0017,False,-170.0,-203.0,-136.3
29,"Calibrated Random Forest (Platt, 5 feats)",-0.0242,-0.0516,0.0017,False,-170.0,-203.0,-136.3


## 6. Phase 4: False Expansion & Feature Interaction Analysis
Inspects whether nonlinear models curb false expansions caused by BM25 $\times$ Redundancy.

In [6]:
df_errors = pd.read_csv(PROJECT_ROOT / "results/week5/model_benchmark/error_analysis.csv")
display(df_errors)

df_feat_imp = pd.read_csv(PROJECT_ROOT / "results/week5/model_benchmark/feature_importance.csv")
print("Permutation Feature Importance Across Families:")
display(df_feat_imp)

,model_name,true_negative_count,true_positive_count,false_expansion_count,false_stop_count,false_expansion_pct,false_stop_pct,true_positive_rate_pct,avg_gain_captured,avg_distractor_penalty
0,"Balanced Logistic Regression (5 feats, Baseline)",120,29,66,16,28.57,6.93,64.44,0.3395,0.4184
1,Balanced Logistic Regression (2 feats: BM25+Lex),117,27,69,18,29.87,7.79,60.00,0.3394,0.3191
2,Shallow Decision Tree (5 feats),128,16,58,29,25.11,12.55,35.56,0.3561,0.3070
3,Random Forest (5 feats),127,25,59,20,25.54,8.66,55.56,0.2487,0.3128
4,XGBoost (5 feats),142,21,44,24,19.05,10.39,46.67,0.2570,0.2441
5,XGBoost (2 feats: BM25+Lex),129,23,57,22,24.68,9.52,51.11,0.3121,0.2671
6,LightGBM (5 feats),118,22,68,23,29.44,9.96,48.89,0.2648,0.3342
7,HistGradientBoosting (5 feats),127,23,59,22,25.54,9.52,51.11,0.2613,0.2494
8,"Small MLP (5->8->1, 5 feats)",130,19,56,26,24.24,11.26,42.22,0.2510,0.3468
9,"Calibrated Logistic Regression (Platt, 5 feats)",186,0,0,45,0.00,19.48,0.00,0.0000,0.0000


Permutation Feature Importance Across Families:


,model_family,feature,permutation_importance_mean,permutation_importance_std
0,logistic_regression,bm25_mean_score,0.0275,0.0248
1,logistic_regression,lex_coverage_top2,0.0701,0.0289
2,logistic_regression,sem_sim_mean_top6,0.0173,0.0164
3,logistic_regression,evidence_query_cluster_span,0.0108,0.0095
4,logistic_regression,evidence_lexical_overlap_mean,0.0009,0.0010
5,decision_tree,bm25_mean_score,0.2511,0.0248
6,decision_tree,lex_coverage_top2,0.0750,0.0106
7,decision_tree,sem_sim_mean_top6,0.0000,0.0000
8,decision_tree,evidence_query_cluster_span,0.1175,0.0286
9,decision_tree,evidence_lexical_overlap_mean,0.1413,0.0269


## 7. Multi-Objective Pareto Frontier Analysis
Maps candidate learned policies against static baselines ($k=2, 4, 5, 6, 8$) and the Oracle sequential ceiling.

In [7]:
df_pareto = pd.read_csv(PROJECT_ROOT / "results/week5/model_benchmark/pareto_points.csv")
display(df_pareto[["system", "mean_f1", "mean_context_tokens", "context_reduction_pct_vs_k8", "mean_regret", "is_pareto_optimal"]])

,system,mean_f1,mean_context_tokens,context_reduction_pct_vs_k8,mean_regret,is_pareto_optimal
0,Static SARA k=2,0.303835,534.5,69.60,0.185672,True
1,Oracle Sequential Ceiling,0.406900,650.3,63.02,0.082500,True
2,Static SARA k=4,0.359601,946.1,46.20,0.129906,False
3,Policy C1 [Calibrated Logistic Regression (Pla...,0.359600,946.1,46.20,0.129900,False
4,"Policy C1 [Calibrated Random Forest (Platt, 5 ...",0.359600,946.1,46.20,0.129900,False
5,"Policy C1 [Calibrated XGBoost (Platt, 5 feats)...",0.359600,946.1,46.20,0.129900,False
6,Policy C1 [XGBoost (5 feats)] (t=0.50),0.372500,1062.3,39.59,0.117000,False
7,Policy C1 [Shallow Decision Tree (5 feats)] (t...,0.367100,1078.4,38.67,0.122400,False
8,"Policy C1 [Small MLP (5->8->1, 5 feats)] (t=0.50)",0.363800,1080.2,38.57,0.125700,False
9,Policy C1 [XGBoost (2 feats: BM25+Lex)] (t=0.50),0.378000,1089.1,38.06,0.111500,False


## 8. Verification of Generated Artifacts

In [8]:
output_dir = PROJECT_ROOT / "results/week5/model_benchmark"
fig_dir = output_dir / "figures"

print("Generated CSV Tables:")
for p in sorted(output_dir.glob("*.csv")):
    print(f"  - {p.name}")

print("\nGenerated Publication Figures:")
for p in sorted(fig_dir.glob("*.png")):
    print(f"  - {p.name}")

Generated CSV Tables:
  - bootstrap_comparison.csv
  - calibration_results.csv
  - error_analysis.csv
  - feature_importance.csv
  - interaction_analysis.csv
  - model_comparison.csv
  - oof_predictions.csv
  - pareto_points.csv
  - policy_comparison.csv
  - subgroup_results.csv
  - threshold_comparison.csv

Generated Publication Figures:
  - bootstrap_comparison.png
  - error_breakdown.png
  - feature_importance.png
  - interaction_analysis.png
  - model_calibration.png
  - model_comparison_bars.png
  - model_policy_pareto.png
  - model_roc_pr.png
  - subgroup_performance.png
  - threshold_quality_cost.png
